In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import tensorflow as tf
import subprocess
import os
import threading
import queue

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            # Enable memory growth so it doesn't seize the whole card
            tf.config.experimental.set_memory_growth(gpu, True)
        print("TensorFlow GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(f"Memory growth error: {e}")

TensorFlow GPU Memory Growth Enabled


In [ ]:
detector = YOLO("yolo26m.pt") 
# classifier = tf.keras.models.load_model('traffic_light_cnn.h5')

In [ ]:
name ='hongkong2'
input_name = 'video/'+name+'.mp4'
base_output = f'output_video/traffic_analysis_output_{name}'
extension = '.mp4'
# output_name = 'output_video/traffic_analysis_output_'+ name+'.mp4'
output_path = f"{base_output}{extension}"
counter = 1

# Check if the file exists and increment a counter until a unique name is found
while os.path.exists(output_path):
    output_path = f"{base_output}_{counter}{extension}"
    counter += 1

cap = cv2.VideoCapture(input_name)
# output_path = output_name

In [ ]:
# 2. Get Video Properties for Saving
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# SET CUSTOM FPS
# output_fps = 2  # This makes 2 frames last 1 second

In [ ]:
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

In [ ]:
# Setup FFmpeg pipe for GPU encoding (NVENC)
# This sends raw frames from Python directly to the GPU encoder
cmd = [
    'ffmpeg',
    '-y',                                # Overwrite output
    '-f', 'rawvideo',                    # Input is raw pixels from Python
    '-vcodec', 'rawvideo',
    '-s', f'{width}x{height}',           # Frame size
    '-pix_fmt', 'bgr24',                 # OpenCV default format
    '-r', str(fps),                      # Framerate
    '-i', '-',                           # Read from stdin (the pipe)
    '-c:v', 'h264_nvenc',                # <--- THE GPU ENCODER

    # --- QUALITY TWEAKS START HERE ---
    '-preset', 'p3',             # p6 or p7 for high quality (slower but cleaner)
    '-tune', 'ull',              # Ultra-Low Latency tuning (skips lookahead)
    '-rc', 'vbr',                # Variable Bitrate mode
    '-cq', '28',                 # The "CRF" equivalent for NVENC (18-23 is the sweet spot)
    '-b:v', '0',                 # Required when using -cq to let it vary freely
    '-profile:v', 'main',        # Use High profile for better compression
    '-spatial-aq', '0',          # Spatial Adaptive Quantization (helps with grain/textures)
    '-temporal-aq', '0',         # Temporal Adaptive Quantization
    # '-bf', '0',                # Disable B-frames (B-frames require "looking ahead")
    # '-delay', '0',               # Force immediate output
    # ---------------------------------
    
    # '-preset', 'fast', 
    '-pix_fmt', 'yuv420p',               # Format for maximum compatibility
    output_path
]

In [ ]:
proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)

In [ ]:
# --- STEP 1: THE BACKGROUND WRITER CLASS ---
class VideoWriterThread(threading.Thread):
    def __init__(self, ffmpeg_process):
        super().__init__()
        self.ffmpeg_process = ffmpeg_process
        self.frame_queue = queue.Queue(maxsize=60) # Buffer to handle spikes
        self.stopped = False

    def run(self):
        while not self.stopped or not self.frame_queue.empty():
            try:
                # Short timeout allows thread to check self.stopped frequently
                frame = self.frame_queue.get(timeout=0.1)
                self.ffmpeg_process.stdin.write(frame.tobytes())
                self.frame_queue.task_done()
            except queue.Empty:
                continue
        
        # Critical Shutdown Sequence
        print("Finishing NVENC encoding...")
        self.ffmpeg_process.stdin.close()
        self.ffmpeg_process.wait()
        print("Video saved successfully.")


    def write(self, frame):
        # This will now BLOCK if the queue is full (maxsize=60).
        # The main loop will wait until the encoder frees up a slot.
        # This guarantees 100% frame integrity.
        self.frame_queue.put(frame.copy(), block=True)

    def stop(self):
        self.stopped = True

In [ ]:
# 2. Define Color Ranges (Hue, Saturation, Value)
# Red has two ranges because it wraps around the 0/180 degree mark
lower_red1 = np.array([0, 100, 100])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([160, 100, 100])
upper_red2 = np.array([180, 255, 255])

lower_yellow = np.array([15, 100, 100])
upper_yellow = np.array([35, 255, 255])

lower_green = np.array([40, 100, 100])
upper_green = np.array([90, 255, 255])

In [ ]:
def get_light_state(crop):
    # 1. Convert to HSV
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    
    

    # 3. Create Masks
    mask_red = cv2.addWeighted(cv2.inRange(hsv, lower_red1, upper_red1), 1.0, 
                               cv2.inRange(hsv, lower_red2, upper_red2), 1.0, 0)
    mask_yellow = cv2.inRange(hsv, lower_yellow, upper_yellow)
    mask_green = cv2.inRange(hsv, lower_green, upper_green)

    # 4. Divide crop into 3 horizontal zones (Top, Middle, Bottom)
    height, _ = mask_red.shape
    top = mask_red[0:height//3, :]
    mid = mask_yellow[height//3:2*height//3, :]
    bot = mask_green[2*height//3:height, :]

    # 5. Determine state based on pixel density in specific zones
    if cv2.countNonZero(top) > 10: # Threshold of 10 pixels
        return "Red"
    elif cv2.countNonZero(mid) > 10:
        return "Yellow"
    elif cv2.countNonZero(bot) > 10:
        return "Green"
    
    return "UNKNOWN"

In [ ]:
writer = VideoWriterThread(proc)
writer.start()

frame_count = 0

# Configuration
CLASS_NAMES = ['Green', 'Red', 'Yellow'] # Ensure this matches your training order
COLOR_MAP = {'Green': (0, 255, 0), 'Red': (0, 0, 255), 'Yellow': (0, 255, 255)}

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
        
    # frame_id = cap.get(cv2.CAP_PROP_POS_FRAMES)
    
    # if frame_id % (fps // 2) != 0:
    #     continue # Skip frames to only process 2 per "real" second

    frame_count += 1

    if frame_count % 3 == 0:

        img_h, img_w, _ = frame.shape
        img_center_x = img_w / 2
    
        # 3. Detection (Class 9 is traffic light)
        results = detector(frame, verbose=False, classes=[9])[0]
        
        lights = results.boxes
        if len(lights) > 0:
            valid_crops = []
            valid_boxes = []
           
    
            # best_box = max(lights, key=lambda b: (b.xyxy[0][2]-b.xyxy[0][0]) * (b.xyxy[0][3]-b.xyxy[0][1]))
            for box in lights:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
        
                # 5. Crop and Classify with your Vertical CNN
                # Note: Ensure crop is not empty
                crop = frame[max(0, y1):y2, max(0, x1):x2]
                if crop.size > 0:
                    # Resize to your new vertical shape (64 width, 64 height)
                    # crop_resized = cv2.resize(crop, (64, 64)) / 255.0
    
                    crop_resized = cv2.resize(crop, (64, 64))  #removed for hsv
    
                    valid_crops.append(crop_resized)
                    valid_boxes.append((x1, y1, x2, y2))
    
            # 2. Perform Batch Inference (The Speed Hack)
            if valid_crops:
    
                # # Convert list to a Tensor (this is faster than np.array for the function call)
                # batch_tensor = tf.convert_to_tensor(valid_crops, dtype=tf.float32)
                
        
                # # Calling the model directly skips the heavy .predict() wrapper
                # predictions = classifier(batch_tensor, training=False).numpy()
    
                # 3. Draw UI
                for i, (x1, y1, x2, y2) in enumerate(valid_boxes):
                    # pred_scores = predictions[i]
                    # color_idx = np.argmax(pred_scores)
                    # conf = pred_scores[color_idx]
                    # color_label = CLASS_NAMES[color_idx]
    
                    color_label = get_light_state(valid_crops[i])
                    
                    bgr_color = COLOR_MAP.get(color_label, (255, 255, 255))
                    cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)

        writer.write(frame)  # Instant! Doesn't wait for FFmpeg
    
   

cap.release()
writer.stop()
writer.join()
# proc.stdin.close()
# proc.wait()
cv2.destroyAllWindows()
print(f"Video saved successfully to {output_path}")

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Finishing NVENC encoding...


frame=79070 fps= 95 q=21.0 Lsize= 2768753kB time=00:21:59.13 bitrate=17194.3kbits/s speed=1.59x    
video:2768405kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.012571%


Video saved successfully.
Video saved successfully to output_video/traffic_analysis_output_hongkong2.mp4
